# 05 — A/B simulado e causal (DiD)

Duas frentes: (1) **A/B simulado** — randomiza países, aplica uplift no proxy UHC e
avalia o efeito com t-test, IC, Cohen's d, poder, CUPED e Bonferroni; (2) **causal** —
event study (UHC → expectativa de vida) e **controle sintético** para países "herói".

In [1]:
import sys; sys.path.insert(0, ".")
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ab = json.load(open("reports/ab_simulation.json"))
causal = json.load(open("reports/causal_did.json"))
ab["config"]

{'delta_uhc': 0.3,
 'fracao_tratamento': 0.5,
 'seed': 42,
 'periodo': '>=2020',
 'n_testes': 2,
 'unidade': 'pais (media do periodo)'}

## 1. A/B simulado — uplift de +0.30 no índice UHC

In [2]:
linhas = []
for metrica, v in ab["metricas"].items():
    linhas.append({
        "desfecho": metrica,
        "efeito": round(v["efeito"], 2),
        "IC95": f"[{v['ic95'][0]:.2f}, {v['ic95'][1]:.2f}]",
        "p": f"{v['p']:.3g}",
        "poder": round(v["poder"], 2),
        "Cohen_d": round(v["cohen_d"], 3),
        "CUPED_efeito": round(v["cuped_efeito"], 2),
        "CUPED_p": f"{v['cuped_p']:.3g}",
        "CUPED_var_reduz": f"{v['cuped_reducao_variancia']:.0%}",
        "p_Bonferroni": f"{v['p_bonferroni']:.3g}",
    })
pd.DataFrame(linhas)

,desfecho,efeito,IC95,p,poder,Cohen_d,CUPED_efeito,CUPED_p,CUPED_var_reduz,p_Bonferroni
0,life_expectancy,2.04,"[-0.20, 4.28]",0.0741,0.43,0.261,1.13,0.0283,80%,0.148
1,child_mortality,-18.80,"[-28.54, -9.06]",0.000191,0.97,-0.552,-16.88,2.11e-10,70%,0.000382


### Leitura

- O A/B no nível de **país** (~108 por braço) tem poder limitado: para um uplift
  realista, seria preciso amostra maior.
- **CUPED** (covariável = basal predito) reduz a variância ~80% e recupera o efeito
  de mortalidade <5; é a alavanca padrão para ganhar poder sem mais amostra.
- A correção de **Bonferroni** (2 desfechos) mantém o controle do erro tipo I.

## 2. Event study (UHC → expectativa de vida)

In [3]:
es = pd.DataFrame(causal["event_study"]["event_study"])
fig = go.Figure()
fig.add_trace(go.Scatter(x=es["k"], y=es["coef"], mode="markers+lines", name="coef"))
fig.add_trace(go.Scatter(
    x=list(es["k"]) + list(es["k"][::-1]),
    y=[c[0] for c in es["ic95"]] + [c[1] for c in es["ic95"]][::-1],
    fill="toself", fillcolor="rgba(0,100,200,0.15)", line=dict(color="rgba(0,0,0,0)"),
    name="IC95%"))
fig.add_vline(x=-1, line_dash="dash", line_color="gray")
fig.add_hline(y=0, line_dash="dot", line_color="black")
fig.update_layout(title="Event study: efeito dinâmico de cruzar o limiar UHC",
                  xaxis_title="anos desde o tratamento", yaxis_title="efeito na LE (anos)")
fig

### Leitura

- Os coeficientes **pré-tratamento** (k<0) já são positivos e grandes → **violação de
  tendências paralelas**: países que cruzam o limiar já vinham ganhando mais LE.
  Isso reforça que o tratamento é **endógeno** (seleção), não um choque aleatório.
- O efeito estimado é, portanto, um limite superior enviesado; o controle sintético
  abaixo quantifica melhor a diferença em relação a um contrafactual plausível.

## 3. Controle sintético

In [4]:
from IPython.display import display
for s in causal["synthetic_control"]:
    d = pd.DataFrame(s["serie"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d["year"], y=d["observado"], name=s["hero"], mode="lines+markers"))
    fig.add_trace(go.Scatter(x=d["year"], y=d["sintetico"], name="sintético", mode="lines+markers",
                             line=dict(dash="dash")))
    fig.add_vline(x=s["treat_year"], line_dash="dash", line_color="gray")
    fig.update_layout(title=f"{s['hero']}: observado vs sintético (pre-RMSE={s['pre_rmse']:.2f}, "
                            f"ATT={s['att_pos']:+.2f})",
                      xaxis_title="ano", yaxis_title="expectativa de vida")
    display(fig)

### Leitura

- O sintético reproduz bem o pré-tratamento (pre-RMSE baixo), validando o contrafactual.
- O **ATT pós-tratamento é negativo** para os dois heróis: o país real cresceu *menos*
  que o sintético. Junto com a convergência vista no F5, isso indica que cruzar o
  limiar UHC **não** acelerou a LE nesses casos — possivelmente por seleção (países já
  em transição) e por o limiar ser um corte arbitrário de um índice contínuo.

## Conclusões

1. O A/B simulado mostra a mecânica completa e o valor do CUPED; o desenho país-nível
   exige efeitos grandes para ter poder.
2. Event study e controle sintético **não** confirmam que cruzar UHC acelera a LE:
   há pré-tendências e ATT negativo.
3. Próximos passos (F8+): levar esses resultados para API/dashboard e monitorar drift.